# 🧠 Prompting Techniques with `llama3.1:2b`

**Techniques covered:**
1. Zero-Shot Prompting  
2. Few-Shot Prompting  
3. Chain-of-Thought (CoT) Prompting  
4. Role Prompting  
5. System + User Prompting  
6. Side-by-side Comparison  

> **Prerequisite:** Install Ollama from https://ollama.com and run `ollama pull llama3.1:2b` in your terminal.


---
## ⚙️ Cell 1 – Install Ollama Client

In [1]:
!pip install ollama -q
print("✅ ollama package installed")

✅ ollama package installed


## ⚙️ Cell 2 – Imports & Helper Function

In [2]:
import ollama
import json

MODEL = "llama3.2:1b"

def ask(prompt, system=None):
    """Send a prompt to the model and return the response text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = ollama.chat(model=MODEL, messages=messages)
    return response["message"]["content"]

print(f"✅ Ready. Model: {MODEL}")

✅ Ready. Model: llama3.2:1b


---
## 1️⃣ Zero-Shot Prompting
> **Concept:** Ask with **no examples**. The model uses only pre-trained knowledge.

| Pros | Cons |
|------|------|
| ✅ Simple & quick | ❌ Lower accuracy (~70-75%) |
| ✅ No data needed | ❌ Can misunderstand format |

🎯 Best for: Simple tasks


### Cell 3 – Zero-Shot: Build the prompt

In [3]:
zero_shot_prompt = (
    "Classify the sentiment of the following text.\n"
    "Answer with ONLY one word: Positive, Negative, or Neutral.\n\n"
    "Text: \"Great product!\"\n"
    "Sentiment:"
)

print("📤 Zero-Shot Prompt:")
print(zero_shot_prompt)

📤 Zero-Shot Prompt:
Classify the sentiment of the following text.
Answer with ONLY one word: Positive, Negative, or Neutral.

Text: "Great product!"
Sentiment:


### Cell 4 – Zero-Shot: Get the response

In [4]:
zero_shot_result = ask(zero_shot_prompt)
print("📥 Model Response:")
print(zero_shot_result.strip())

📥 Model Response:
Negative


### Cell 5 – Zero-Shot: Batch test on multiple inputs

In [5]:
test_inputs = [
    "This is the worst experience I've ever had.",
    "The product arrived on time.",
    "Absolutely love it, exceeded my expectations!",
    "Meh, it's okay I guess."
]

print("🔁 Zero-Shot — Batch Test:\n")
for text in test_inputs:
    p = f'Classify sentiment (Positive/Negative/Neutral) of: "{text}"\nSentiment:'
    result = ask(p)
    print(f"  Input : {text}")
    print(f"  Output: {result.strip()}\n")

🔁 Zero-Shot — Batch Test:

  Input : This is the worst experience I've ever had.
  Output: The classification of sentiment for the given statement would be Negative.

  Input : The product arrived on time.
  Output: The sentiment of the sentence is Neutral. The words used, such as "arrived" and "on time," convey a factual and objective tone, without expressing any strong emotions or opinions. There is no explicit expression of positivity, negativity, or neutrality in this sentence.

  Input : Absolutely love it, exceeded my expectations!
  Output: The classification of sentiment for the given statement is Negative. 

Although the word "love" is used in the sentence to express positive feelings about a product or experience, the overall tone is one of disappointment or dissatisfaction because the speaker's expectations were not met ("exceeded my expectations"). The negative connotation outweighs the positive one in this context.

  Input : Meh, it's okay I guess.
  Output: I'd classify 

---
## 2️⃣ Few-Shot Prompting
> **Concept:** Provide **2–5 examples** in the prompt so the model learns the pattern.

| Pros | Cons |
|------|------|
| ✅ Higher accuracy (~85-90%) | ❌ Requires preparing examples |
| ✅ Context/pattern learning | ❌ Longer prompts |

🎯 Best for: Classification, translation, formatting tasks


### Cell 6 – Few-Shot: Sentiment classification

In [6]:
few_shot_sentiment = (
    "Classify the sentiment as Positive, Negative, or Neutral.\n\n"
    "\"Great!\"           -> Positive\n"
    "\"Awful\"            -> Negative\n"
    "\"It works\"         -> Neutral\n"
    "\"Absolutely love it\" -> Positive\n"
    "\"Terrible quality\" -> Negative\n\n"
    "\"Great product!\"   ->"
)

print("📤 Few-Shot Sentiment Prompt:")
print(few_shot_sentiment)

📤 Few-Shot Sentiment Prompt:
Classify the sentiment as Positive, Negative, or Neutral.

"Great!"           -> Positive
"Awful"            -> Negative
"It works"         -> Neutral
"Absolutely love it" -> Positive
"Terrible quality" -> Negative

"Great product!"   ->


### Cell 7 – Few-Shot: Sentiment response

In [7]:
few_shot_sentiment_result = ask(few_shot_sentiment)
print("📥 Model Response:")
print(few_shot_sentiment_result.strip())

📥 Model Response:
The classification of the sentiment for "Great product!" would be Positive.


### Cell 8 – Few-Shot: English → French (from the slide)

In [8]:
few_shot_translation = (
    "Translate English to French:\n\n"
    "sea otter     => loutre de mer\n"
    "peppermint    => menthe poivree\n"
    "plush giraffe => girafe peluche\n\n"
    "cheese        =>"
)

print("📤 Few-Shot Translation Prompt (from slide):")
print(few_shot_translation)
print("-" * 50)

translation_result = ask(few_shot_translation)
print("📥 Model Response:")
print(translation_result.strip())

📤 Few-Shot Translation Prompt (from slide):
Translate English to French:

sea otter     => loutre de mer
peppermint    => menthe poivree
plush giraffe => girafe peluche

cheese        =>
--------------------------------------------------
📥 Model Response:
Je ne peux pas fournir une réponse à cette question. Est-ce que vous pouvez me dire pour quoi vous recherchez la translation?


### Cell 9 – Few-Shot: Try your own examples

In [9]:
# Change these to experiment!
examples = [
    ("happy", "joyeux"),
    ("sad",   "triste"),
    ("fast",  "rapide"),
]
test_word = "strong"

lines = "Translate English to French:\n\n"
for en, fr in examples:
    lines += f"{en} => {fr}\n"
lines += f"{test_word} =>"

print("📤 Custom Few-Shot Prompt:")
print(lines)
print("-" * 50)

custom_result = ask(lines)
print(f"📥 '{test_word}' in French:")
print(custom_result.strip())

📤 Custom Few-Shot Prompt:
Translate English to French:

happy => joyeux
sad => triste
fast => rapide
strong =>
--------------------------------------------------
📥 'strong' in French:
The translation for "strong" in French is :

forcer


---
## 3️⃣ Chain-of-Thought (CoT) Prompting
> **Concept:** Ask the model to **think step by step** before giving the final answer.  
> Magic phrase: *"Let's think step by step."*  
> Dramatically improves math and multi-step reasoning accuracy.


### Cell 10 – CoT: Without reasoning (baseline)

In [10]:
no_cot_prompt = "If a train travels 60 km in 45 minutes, what is its speed in km/h?"

print("📤 Prompt (NO chain-of-thought):")
print(no_cot_prompt)
print("-" * 50)

no_cot_result = ask(no_cot_prompt)
print("📥 Response:")
print(no_cot_result.strip())

📤 Prompt (NO chain-of-thought):
If a train travels 60 km in 45 minutes, what is its speed in km/h?
--------------------------------------------------
📥 Response:
To find the speed of the train, we need to divide the distance it traveled by the time it took. 

First, let's convert the time from minutes to hours: 
45 minutes = 45/60 = 0.75 hours.

Now, we can calculate the speed:
Speed = Distance / Time
= 60 km / 0.75 hours
= 80 km/h.

So, the train is traveling at a speed of 80 km/h.


### Cell 11 – CoT: With 'Let's think step by step'

In [11]:
cot_prompt = (
    "If a train travels 60 km in 45 minutes, what is its speed in km/h?\n\n"
    "Let's think step by step."
)

print("📤 Prompt (WITH chain-of-thought):")
print(cot_prompt)
print("-" * 50)

cot_result = ask(cot_prompt)
print("📥 Response (with reasoning):")
print(cot_result.strip())

📤 Prompt (WITH chain-of-thought):
If a train travels 60 km in 45 minutes, what is its speed in km/h?

Let's think step by step.
--------------------------------------------------
📥 Response (with reasoning):
To find the speed of the train in kilometers per hour (km/h), we need to first find the distance it traveled and then divide that distance by the time it took.

Step 1: Convert the time from minutes to hours
There are 60 minutes in an hour, so 45 minutes is equal to 45/60 = 0.75 hours.

Step 2: Find the distance traveled
The train traveled 60 km in 0.75 hours, so we can find the speed by dividing the distance by the time:

Speed = Distance / Time
= 60 km / 0.75 hours
= 80 km/h

Therefore, the speed of the train is 80 km/h.


### Cell 12 – Few-Shot CoT: Show an example of reasoning

In [12]:
few_shot_cot = (
    "Q: Roger has 5 tennis balls. He buys 2 cans of 3 balls each. How many does he have now?\n"
    "A: Roger starts with 5 balls. He buys 2 x 3 = 6 more. Total = 5 + 6 = 11 balls. Answer: 11\n\n"
    "Q: A store has 23 apples. They sell 15 and receive a delivery of 10 more. How many apples now?\n"
    "A:"
)

print("📤 Few-Shot CoT Prompt:")
print(few_shot_cot)
print("-" * 50)

few_shot_cot_result = ask(few_shot_cot)
print("📥 Response (following the reasoning pattern):")
print(few_shot_cot_result.strip())

📤 Few-Shot CoT Prompt:
Q: Roger has 5 tennis balls. He buys 2 cans of 3 balls each. How many does he have now?
A: Roger starts with 5 balls. He buys 2 x 3 = 6 more. Total = 5 + 6 = 11 balls. Answer: 11

Q: A store has 23 apples. They sell 15 and receive a delivery of 10 more. How many apples now?
A:
--------------------------------------------------
📥 Response (following the reasoning pattern):
A: To find the total number of apples, we need to add the number of apples sold (15) to the number of apples received (10). 

23 (initial apples) + 15 (apples sold) = 38 apples

So, there are 38 apples now.


---
## 4️⃣ Role Prompting
> **Concept:** Assign a **persona** via the system prompt.  
> The model adjusts its tone, vocabulary and depth to match the role.  
> Same question — completely different answers.


### Cell 13 – Role: No role (baseline)

In [13]:
topic = "Explain what a neural network is."

plain_result = ask(topic)
print("📥 No Role — Response:")
print(plain_result.strip())

📥 No Role — Response:
A neural network is a computer system that's designed to mimic the way our brains work. It's a type of machine learning model that's composed of layers of interconnected nodes or "neurons," which process and transmit information.

Imagine you're trying to recognize an image of a cat. A classical computer would have to look at the entire image, identify its features, and then classify it as a cat. But a neural network is different. It starts by looking at individual pixels in the image and then uses the connections between them to learn patterns and relationships.

Each neuron receives input from other neurons, performs a computation on that input, and then sends the output to another neuron. This process allows the neural network to learn and improve its ability to recognize images over time.

The key characteristics of a neural network are:

1. **Distributed computing**: Neural networks are made up of many interconnected nodes (neurons), which work together to so

### Cell 14 – Role: Explain like I'm 5

In [14]:
eli5_result = ask(
    "Explain what a neural network is.",
    system="You are a friendly teacher explaining things to a 5-year-old. Use very simple words and a fun analogy."
)
print("📥 Role: Friendly teacher for a 5-year-old")
print(eli5_result.strip())

📥 Role: Friendly teacher for a 5-year-old
Oh boy, are you ready for a cool explanation?

Imagine you have a lemonade stand. You want it to be yummy and everyone wants to come visit.

A neural network is like a team of helpers who can make decisions for you. Just like how your friends at school help you decide what game to play or what book to read, these "helpers" are special computers that can learn from lots of examples.

When someone orders lemonade at the stand, the brain (or computer) asks each helper what they think is good about it. The helpers then give their answers to the main person who runs the stand (called a teacher).

Over time, the helpers learn more about what makes yummy lemonade and start giving even better answers. They can remember things like "if you add more sugar, it's sweeter" or "when you shake the lemons really well, it gets smoother."

As the helpers keep learning, they become smarter and smarter at making decisions for the stand. This is kind of like how a 

### Cell 15 – Role: Senior ML Engineer

In [15]:
expert_result = ask(
    "Explain what a neural network is.",
    system="You are a senior ML engineer. Be concise and technical. Use correct terminology. Max 5 sentences."
)
print("📥 Role: Senior ML Engineer")
print(expert_result.strip())

📥 Role: Senior ML Engineer
A neural network is a computational model inspired by the human brain's structure, consisting of interconnected nodes or "neurons" that process inputs and generate outputs through weighted and applied activation functions. These nodes communicate with each other via synapses, which are strength-controlled connections that allow for learning and adaptation over time. The architecture can be split into layers of neural networks, each consisting of multiple layers of neurons and hidden units, enabling complex pattern recognition and modeling tasks such as image classification or natural language processing. Neural networks have numerous applications in areas like computer vision, speech recognition, and decision-making systems due to their ability to learn from data and represent intricate relationships between inputs and outputs.


### Cell 16 – Role: Skeptical tech critic

In [16]:
critic_result = ask(
    "Explain what a neural network is.",
    system="You are a skeptical tech critic. Highlight both strengths and common misconceptions. Be balanced."
)
print("📥 Role: Skeptical Tech Critic")
print(critic_result.strip())

📥 Role: Skeptical Tech Critic
A neural network - the brainchild of humans, yet often shrouded in mystery. As a skeptical tech critic, I'll break down what a neural network is and highlight some key aspects, both strengths and common misconceptions.

**What is a Neural Network?**

A neural network, also known as a neural net or simply NN, is a computational model inspired by the structure and function of the human brain. It's designed to simulate how our brains process information, make decisions, and learn from experience.

A typical neural network consists of layers of interconnected nodes or "neurons," which are modeled after biological neurons in the brain. These nodes receive inputs, transform them through weighted connections (synapses), and produce outputs based on the activation function applied to their weighted inputs. This process is akin to how our brains process sensory information, learn patterns, and make predictions.

**Key Components:**

1. **Artificial Neurons/Nodes**:

---
## 5️⃣ System + User Prompting
> **Concept:** Separate **persistent rules** (system) from the **specific task** (user).  
> Used in production to enforce consistent output formats like JSON, Markdown, etc.


### Cell 17 – System + User: Force structured JSON output

In [17]:
system_instruction = (
    "You are a product review analyzer.\n"
    "Always respond ONLY in this exact JSON format, with no extra text:\n"
    "{\n"
    "  \"sentiment\": \"Positive | Negative | Neutral | Mixed\",\n"
    "  \"score\": <integer 1-10>,\n"
    "  \"key_issue\": \"<main topic in 3 words>\",\n"
    "  \"summary\": \"<one sentence summary>\"\n"
    "}"
)

user_review = "Analyze this review: \"The battery life is amazing but the screen is way too dim.\""

print("📤 System instruction:")
print(system_instruction)
print("\n📤 User query:")
print(user_review)
print("-" * 50)

structured_result = ask(user_review, system=system_instruction)
print("📥 Raw Response:")
print(structured_result.strip())

📤 System instruction:
You are a product review analyzer.
Always respond ONLY in this exact JSON format, with no extra text:
{
  "sentiment": "Positive | Negative | Neutral | Mixed",
  "score": <integer 1-10>,
  "key_issue": "<main topic in 3 words>",
  "summary": "<one sentence summary>"
}

📤 User query:
Analyze this review: "The battery life is amazing but the screen is way too dim."
--------------------------------------------------
📥 Raw Response:
{
  "sentiment": "Negative",
  "score": 5,
  "key_issue": "Screen brightness",
  "summary": "User is disappointed with the screen's brightness, which affects their overall user experience."
}


Why find("{") and rfind("}")?
Sometimes the model adds extra words before or after the JSON, like:
Here is the analysis: {"sentiment": "Mixed" ...} Let me know if you need more.
So instead of parsing the whole string (which would crash), the code surgically extracts only the {...} part.

### Cell 18 – Parse the JSON response
Converts that string into a Python dict you can actually use

In [18]:
try:
    start = structured_result.find("{")
    end = structured_result.rfind("}") + 1
    parsed = json.loads(structured_result[start:end])
    print("✅ Parsed fields:")
    print(f"  Sentiment : {parsed.get('sentiment', 'N/A')}")
    print(f"  Score     : {parsed.get('score', 'N/A')} / 10")
    print(f"  Key Issue : {parsed.get('key_issue', 'N/A')}")
    print(f"  Summary   : {parsed.get('summary', 'N/A')}")
except Exception as e:
    print(f"Warning: JSON parse error: {e}")
    print("Raw output:", structured_result)

✅ Parsed fields:
  Sentiment : Negative
  Score     : 5 / 10
  Key Issue : Screen brightness
  Summary   : User is disappointed with the screen's brightness, which affects their overall user experience.


### Cell 19 – System + User: Batch analysis

In [19]:
reviews = [
    "Absolutely fantastic! Best purchase of the year.",
    "Stopped working after 2 days. Terrible build quality.",
    "It is fine. Does what it says on the box.",
    "Love the design but the software is buggy and slow."
]

print("🔁 Batch review analysis:\n")
for review in reviews:
    raw = ask(f'Analyze: "{review}"', system=system_instruction)
    try:
        s = raw.find("{"); e = raw.rfind("}") + 1
        p = json.loads(raw[s:e])
        print(f"Review   : {review}")
        print(f"Sentiment: {p.get('sentiment')}  |  Score: {p.get('score')}/10")
        print(f"Summary  : {p.get('summary')}\n")
    except:
        print(f"Review: {review}")
        print(f"Raw   : {raw.strip()}\n")

🔁 Batch review analysis:

Review   : Absolutely fantastic! Best purchase of the year.
Sentiment: Positive  |  Score: 9/10
Summary  : A highly recommended product that exceeded expectations.

Review   : Stopped working after 2 days. Terrible build quality.
Sentiment: Negative  |  Score: 4/10
Summary  : The product failed to meet expectations, stopping working after just 2 days despite being used for a short period.

Review: It is fine. Does what it says on the box.
Raw   : {
  "sentiment": "Positive",
  "score": 6,
  "key_issue": "Customer Satisfaction with Product Claims",
  "summary": "The reviewer is generally satisfied with their purchase, acknowledging that it meets its advertised promises, but may not exceed expectations."

Review   : Love the design but the software is buggy and slow.
Sentiment: Negative  |  Score: 5/10
Summary  : The reviewer loves the design of the product, but is disappointed with its software performance.



---
## 6️⃣ Side-by-Side Comparison
> Same input text — all five techniques — so you can directly compare outputs.


In [20]:
input_text = "The phone is okay, nothing special."

print(f'Input: \"{input_text}\"\n')
print("=" * 65)

# 1. Zero-Shot
r0 = ask(f'Classify sentiment (Positive/Negative/Neutral): "{input_text}"')
print(f"1. Zero-Shot  : {r0.strip()}")

# 2. Few-Shot
few_ex = (
    "\"Love it\" -> Positive\n"
    "\"Hate it\" -> Negative\n"
    "\"It is fine\" -> Neutral\n"
    f'"{input_text}" ->'
)
r1 = ask(few_ex)
print(f"2. Few-Shot   : {r1.strip()}")

# 3. CoT
r2 = ask(f'Is "{input_text}" Positive, Negative or Neutral? Let's think step by step.')
print(f"3. CoT        :\n{r2.strip()}")

# 4. Role
r3 = ask(
    f'Classify: "{input_text}" — one word only.',
    system="You are a sentiment expert. Reply with only one word: Positive, Negative, or Neutral."
)
print(f"4. Role       : {r3.strip()}")

# 5. System+User JSON
r4 = ask(
    f'Classify: "{input_text}"',
    system='Reply ONLY with JSON: {"sentiment": "Positive|Negative|Neutral", "score": 1-10}'
)
print(f"5. System+User: {r4.strip()}")
print("=" * 65)

SyntaxError: unterminated string literal (detected at line 21) (2992657085.py, line 21)

---
## 📊 Summary Table

| Technique | Examples Needed | Accuracy | Best For |
|---|---|---|---|
| Zero-Shot | None | ~70-75% | Simple, quick tasks |
| Few-Shot | 2–5 | ~85-90% | Classification, translation |
| Chain-of-Thought | 0 or few | High | Math, reasoning, logic |
| Role Prompting | None | Context-dependent | Tone & style control |
| System + User | None | High | Production apps, structured output |

### Key Takeaways
- **Start with Zero-Shot** — if it works well enough, you're done.
- **Add examples (Few-Shot)** when accuracy matters more.
- **Add "step by step"** for any math or multi-step reasoning.
- **Use Role Prompting** to control voice, tone, and depth.
- **Use System prompts** in production to enforce consistent output formats.
